In [25]:
import kagglehub

path = kagglehub.dataset_download("clmentbisaillon/fake-and-real-news-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'fake-and-real-news-dataset' dataset.
Path to dataset files: /kaggle/input/fake-and-real-news-dataset


In [26]:
import pandas as pd

fake_news_path = path + '/Fake.csv'
true_news_path = path + '/True.csv'

fake_news_df = pd.read_csv(fake_news_path)
true_news_df = pd.read_csv(true_news_path)

print("### Initial inspection of Fake News DataFrame ###\n")
print("Fake News DataFrame - Head:")
print(fake_news_df.head())

print("\nFake News DataFrame - Info:")
fake_news_df.info()

print("\nFake News DataFrame - Describe:")
print(fake_news_df.describe(include='all'))

print("\nFake News DataFrame - Missing Values:")
print(fake_news_df.isnull().sum())

print("\n\n### Initial inspection of True News DataFrame ###\n")
print("True News DataFrame - Head:")
print(true_news_df.head())

print("\nTrue News DataFrame - Info:")
true_news_df.info()

print("\nTrue News DataFrame - Describe:")
print(true_news_df.describe(include='all'))

print("\nTrue News DataFrame - Missing Values:")
print(true_news_df.isnull().sum())

### Initial inspection of Fake News DataFrame ###

Fake News DataFrame - Head:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  

Fake News DataFrame - Info:
<class 'pandas.core.frame.DataFr

In [20]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

ps = PorterStemmer()

stop_words_set = set(stopwords.words('english'))

def preprocess_text(text):
    text = text.lower()
    text = re.sub('[^a-zA-Z]', ' ', text)
    words = text.split()
    words = [ps.stem(word) for word in words if word not in stop_words_set]
    return ' '.join(words)

news_df['processed_text'] = news_df['text'].apply(preprocess_text)

print("### News DataFrame after Text Preprocessing - Head ###")
print(news_df[['title', 'text', 'processed_text', 'label']].head())

print("\n### News DataFrame after Text Preprocessing - Info ###")
news_df.info()

### News DataFrame after Text Preprocessing - Head ###
                                               title  \
0   BREAKING: GOP Chairman Grassley Has Had Enoug...   
1   Failed GOP Candidates Remembered In Hilarious...   
2   Mike Pence’s New DC Neighbors Are HILARIOUSLY...   
3  California AG pledges to defend birth control ...   
4  AZ RANCHERS Living On US-Mexico Border Destroy...   

                                                text  \
0  Donald Trump s White House is in chaos, and th...   
1  Now that Donald Trump is the presumptive GOP n...   
2  Mike Pence is a huge homophobe. He supports ex...   
3  SAN FRANCISCO (Reuters) - California Attorney ...   
4  Twisted reasoning is all that comes from Pelos...   

                                      processed_text  label  
0  donald trump white hous chao tri cover russia ...      1  
1  donald trump presumpt gop nomine time rememb c...      1  
2  mike penc huge homophob support ex gay convers...      1  
3  san francisco reuter

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer()

X_tfidf = tfidf_vectorizer.fit_transform(news_df['processed_text'])

print("Shape of TF-IDF feature matrix (X_tfidf):", X_tfidf.shape)

Shape of TF-IDF feature matrix (X_tfidf): (44898, 89633)


In [22]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, news_df['label'], test_size=0.2, random_state=42, stratify=news_df['label']
)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

nb_model = MultinomialNB()
lr_model = LogisticRegression(max_iter=1000, random_state=42)

print("\n Cross-Validation Results ")

nb_cv_scores = cross_val_score(nb_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"Multinomial Naive Bayes - Mean CV Accuracy: {nb_cv_scores.mean():.4f}")

lr_cv_scores = cross_val_score(lr_model, X_train, y_train, cv=5, scoring='accuracy')
print(f"Logistic Regression - Mean CV Accuracy: {lr_cv_scores.mean():.4f}")

Shape of X_train: (35918, 89633)
Shape of X_test: (8980, 89633)
Shape of y_train: (35918,)
Shape of y_test: (8980,)

 Cross-Validation Results 
Multinomial Naive Bayes - Mean CV Accuracy: 0.9332
Logistic Regression - Mean CV Accuracy: 0.9844


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

nb_model.fit(X_train, y_train)
lr_model.fit(X_train, y_train)

nb_predictions = nb_model.predict(X_test)
lr_predictions = lr_model.predict(X_test)

print("\n Model Evaluation ")

print("\n### Multinomial Naive Bayes Model Performance ###")
print("Classification Report:")
print(classification_report(y_test, nb_predictions))

nb_cm = confusion_matrix(y_test, nb_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(nb_cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Multinomial Naive Bayes Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

print("\n### Logistic Regression Model Performance ###")
print("Classification Report:")
print(classification_report(y_test, lr_predictions))

lr_cm = confusion_matrix(y_test, lr_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(lr_cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Logistic Regression Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

In [24]:
import pandas as pd

feature_names = tfidf_vectorizer.get_feature_names_out()

coefficients = lr_model.coef_[0]
coef_series = pd.Series(coefficients, index=feature_names)

print("\n Top Discriminative Words for Logistic Regression ")

top_fake_words = coef_series.nlargest(10)
print("\nTop 10 words indicative of 'Fake News' (label 1):")
print(top_fake_words)

top_real_words = coef_series.nsmallest(10)
print("\nTop 10 words indicative of 'Real News' (label 0):")
print(top_real_words)


 Top Discriminative Words for Logistic Regression 

Top 10 words indicative of 'Fake News' (label 1):
via        10.821189
us          7.428950
imag        7.060920
read        5.864381
mr          5.580399
gop         5.222182
com         4.987539
featur      4.770544
hillari     4.652439
even        4.460038
dtype: float64

Top 10 words indicative of 'Real News' (label 0):
reuter       -26.328706
said         -18.790985
washington    -6.719464
wednesday     -5.766023
tuesday       -5.202505
thursday      -5.186815
friday        -4.753667
minist        -4.293907
monday        -4.286107
nov           -4.216509
dtype: float64
